---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas10_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan seluruh streaming query dihentikan (`query.stop()`) di akhir notebook sebelum dikumpulkan.
- Seluruh cell wajib sudah dijalankan (**Run All**) — perhatikan penggunaan `time.sleep()` yang cukup agar output streaming sempat terisi sebelum notebook mencoba menampilkannya.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Streaming Source | Skema & `readStream` didefinisikan dengan benar | 15% |
| B. Agregasi per Cabang | Kolom turunan & agregasi `groupBy` benar, query berjalan | 20% |
| C. Window Waktu | Agregasi window 2 menit terdefinisi & berjalan dengan benar | 20% |
| D. Pengujian Multi-Batch | Minimal 3 batch terkirim, hasil akumulasi terbukti benar | 25% |
| E. File Sink | Streaming ke Parquet berhasil, terverifikasi dapat dibaca kembali | 20% |


In [68]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
from datetime import datetime
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [69]:
skema_tugas = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

folder_tugas = "tugas10_stream"
for f in [folder_tugas, "tugas10_output", "tugas10_checkpoint"]:
    shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

print("Siap untuk latihan. Folder:", folder_tugas)

Siap untuk latihan. Folder: tugas10_stream


In [70]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

print("df_stream.isStreaming:", df_stream.isStreaming)

df_stream.isStreaming: True


**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [71]:
df_stream2 = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_stream = df_stream2.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)   

print("Transformasi berhasil didefinisikan.")

Transformasi berhasil didefinisikan.


In [72]:
query = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query dimulai! Status:", query.status)

Streaming query dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [73]:
df_stream3 = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

agregasi_window = df_stream3.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_harga")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query window waktu dimulai.")

Streaming query window waktu dimulai.


**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [74]:
#BATCH 1
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ['P1', 'P2', 'P3'],
    "nama_cabang": ['Magelang', 'Solo', 'Magelang'],
    "nama_menu": ['Kopi Susu', 'Americano', 'Latte'],
    "qty": [2, 1, 3],
    "harga_satuan": [18000, 20000, 25000],
    "waktu_pesan": [waktu_sekarang] * 3,
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("select * from agregasi_cabang order by nama_cabang").show()

Batch 1 dikirim. Menunggu Spark memproses...
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     111000|             2|
|       Solo|      20000|             1|
+-----------+-----------+--------------+



In [75]:
#BATCH 2
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ['P4', 'P5'],
    "nama_cabang": ['Magelang', 'Yogyakarta'],
    "nama_menu": ['Americano', 'Kopi Susu'],
    "qty": [2, 4],
    "harga_satuan": [20000, 18000],
    "waktu_pesan": [waktu_sekarang] * 2,
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("select * from agregasi_cabang order by nama_cabang").show()

Batch 2 dikirim. Menunggu Spark memproses...
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     151000|             3|
|       Solo|      20000|             1|
| Yogyakarta|      72000|             1|
+-----------+-----------+--------------+



In [76]:
#BATCH 3
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ['P6', 'P7', 'P8'],
    "nama_cabang": ['Solo', 'Semarang', 'Yogyakarta'],
    "nama_menu": ['Latte', 'Kopi Susu', 'Americano'],
    "qty": [1, 2, 2],
    "harga_satuan": [25000, 18000, 20000],
    "waktu_pesan": [waktu_sekarang] * 3,
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("select * from agregasi_cabang order by nama_cabang").show()

Batch 3 dikirim. Menunggu Spark memproses...
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Magelang|     151000|             3|
|   Semarang|      36000|             1|
|       Solo|      45000|             2|
| Yogyakarta|     112000|             2|
+-----------+-----------+--------------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [77]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query 3 (Parquet sink) berjalan:", query3.isActive)

Query 3 (Parquet sink) berjalan: True


In [78]:
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
pd.DataFrame({
    "pesanan_id": ["P9"],
    "nama_cabang": ["Semarang"],
    "nama_menu": ["Latte"],
    "qty": [2],
    "harga_satuan": [25000],
    "waktu_pesan": [waktu_sekarang],
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark memproses...")
time.sleep(10)  

Batch 4 dikirim. Menunggu Spark memproses...


In [80]:
!ls tugas10_output

part-00000-289b4de8-8b4a-4bcf-968e-deebaf110147-c000.snappy.parquet
part-00001-5aaf8791-d7e6-44a0-94a7-92e39378d524-c000.snappy.parquet
part-00002-84d2152f-3bdb-4884-943f-120c7275e4b0-c000.snappy.parquet
part-00003-93c64f6c-2ad2-49f8-847f-1e7df8d9d914-c000.snappy.parquet
_spark_metadata


In [81]:
df_parquet = spark.read.parquet("tugas10_output")
print("df_parquet.isStreaming:", df_parquet.isStreaming)
print("Jumlah baris:", df_parquet.count(), "(harapan: 9 pesanan dari batch 1-4)")
df_parquet.orderBy("pesanan_id").show(truncate=False)

df_parquet.isStreaming: False
Jumlah baris: 9 (harapan: 9 pesanan dari batch 1-4)
+----------+-----------+---------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu|qty|harga_satuan|waktu_pesan        |
+----------+-----------+---------+---+------------+-------------------+
|P1        |Magelang   |Kopi Susu|2  |18000       |2026-11-05 21:12:00|
|P2        |Solo       |Americano|1  |20000       |2026-11-05 21:12:00|
|P3        |Magelang   |Latte    |3  |25000       |2026-11-05 21:12:00|
|P4        |Magelang   |Americano|2  |20000       |2026-11-05 21:12:06|
|P5        |Yogyakarta |Kopi Susu|4  |18000       |2026-11-05 21:12:06|
|P6        |Solo       |Latte    |1  |25000       |2026-11-05 21:12:13|
|P7        |Semarang   |Kopi Susu|2  |18000       |2026-11-05 21:12:13|
|P8        |Yogyakarta |Americano|2  |20000       |2026-11-05 21:12:13|
|P9        |Semarang   |Latte    |2  |25000       |2026-11-05 21:12:19|
+----------+-----------+---------+---+------------+---

In [82]:
query.stop()
query2.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")

Seluruh streaming query dihentikan, SparkSession ditutup.
